# Final Report: Agentic AI for Proactive QoS Assurance in 6G Network Slicing

This notebook is the project's write-up — it does not introduce any new code. Every number below is reproduced live from `src/*.py` (same calibrated scenario used throughout `02_digital_twin.ipynb` → `06_baseline_comparison.ipynb`), not copy-pasted from an earlier run, so it stays honest if anything upstream changes.


## 1. What Was Built

| Stage | Deliverable | Status |
|---|---|---|
| 1 | Non-linear congestion-cliff Digital Twin, correlated fading, shared contention | Done — `src/digital_twin.py`, 10 tests |
| 2 | Probabilistic LSTM forecaster (mean + variance, Gaussian NLL) | Done — `src/forecaster.py`, 7 tests |
| 3 | Live agentic planner: prompt contract, structured output, bounded retry loop | Done — `src/agent_planner.py`, 9 tests |
| 4 | Two-stage safety layer + deterministic fallback | Done — `src/safety_layer.py`, 14 tests (9 core + 5 adversarial) |
| 5 | Static + legacy-reactive baselines, three-arm comparison harness | Done — `src/baseline_agents.py`, 5 tests |
| 6 | Full three-way empirical comparison | Done — `notebooks/06_baseline_comparison.ipynb` |

**43 tests pass across the whole repository.** Run with `PYTHONPATH=src pytest tests/ -v` from the repo root.


In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
from digital_twin import DigitalTwin, SliceTrafficSpec
from schemas import NetworkRules
from baseline_agents import StaticBaselineAgent, ReactiveLegacyAgent, run_comparison
from reference_llm import build_reference_llm
from live_pipeline import make_live_agentic_controller

RULES = NetworkRules(total_capacity_mbps=100.0, urllc_min_guarantee_mbps=30.0, max_step_change_mbps=20.0)
SPECS = [
    SliceTrafficSpec("URLLC", base_demand_mbps=22.0, noise_std_mbps=1.5,
                      spike_probability=0.03, spike_multiplier=1.9, spike_decay=0.65),
    SliceTrafficSpec("eMBB", base_demand_mbps=45.0, noise_std_mbps=6.0,
                      spike_probability=0.03, spike_multiplier=2.2, spike_decay=0.7),
]

def make_twin():
    return DigitalTwin(SPECS, fading_rho=0.9, fading_min_fraction=0.8, contention_strength=0.12, seed=7)

controllers = {
    "Static Baseline": StaticBaselineAgent({"URLLC": 45.0, "eMBB": 60.0}).decide,
    "Legacy Reactive Agent": ReactiveLegacyAgent().decide,
    "Live Agentic System": make_live_agentic_controller(build_reference_llm()),
}
results = run_comparison(make_twin, controllers, RULES, num_steps=400)
print("Reproduced live for this report.")


## 2. Headline Result


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = list(results.keys())
violation_rates = [results[n]["qos_violation_rate"] * 100 for n in names]
colors = ["#7f8c8d", "#e67e22", "#1f4e8c"]
bars = ax.bar(names, violation_rates, color=colors)
ax.set_ylabel("QoS violation rate (%)")
ax.set_title("URLLC QoS violation rate, 400 timesteps, identical traffic across all three arms")
ax.tick_params(axis='x', rotation=15)
ax.grid(alpha=0.3, axis='y')
for bar, v in zip(bars, violation_rates):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()

static_v = results["Static Baseline"]["qos_violation_rate"]
agentic_v = results["Live Agentic System"]["qos_violation_rate"]
reactive_v = results["Legacy Reactive Agent"]["qos_violation_rate"]
embb_cost = results["Static Baseline"]["mean_embb_throughput_mbps"] - results["Live Agentic System"]["mean_embb_throughput_mbps"]

print(f"\nThe Live Agentic System cut the QoS violation rate from {static_v:.1%} (static baseline) "
      f"to {agentic_v:.1%} -- a {(1 - agentic_v/static_v):.0%} reduction -- and from {reactive_v:.1%} "
      f"(legacy reactive agent) to {agentic_v:.1%}, a {(1 - agentic_v/reactive_v):.0%} reduction.")
print(f"This came at a real, honest cost of {embb_cost:.1f} Mbps "
      f"({embb_cost/results['Static Baseline']['mean_embb_throughput_mbps']:.1%}) of mean eMBB throughput.")


## 3. Safety Layer: Zero Unsafe Plans Reached Actuation

Across every adversarial test in `05_safety_layer.ipynb` — 19 individually malformed/hostile plan attempts, plus a chaotic worst-case run where every retry attempt was also garbage — the safety layer's two invariants held without exception:

- No malformed or constraint-violating plan was ever accepted.
- Every worst-case run correctly fell back to the deterministic safe policy, which by construction always respects the URLLC minimum guarantee and total capacity.

This matters for interpreting Section 2's numbers correctly: the Live Agentic System's improvement did not come at the cost of any safety guarantee being relaxed or bypassed.


## 4. Limitations, Stated Plainly

- **The reference LLM is not a real LLM.** All results above use `src/reference_llm.py`'s deterministic, hand-tuned heuristic, not genuine model reasoning. `real_anthropic_llm_call()` is wired and ready (`notebooks/04_agentic_planner.ipynb`, Section 1) but was not run for this report — these numbers are a *lower bound* on what real LLM reasoning might achieve, not an upper bound.
- **The "legacy PPO agent" is a behavioural stand-in**, not the original trained model from the earlier semester's prototype — see the honesty note in `src/baseline_agents.py`. If the original trained checkpoint still exists, swapping it in only requires matching `ControllerFn`'s signature.
- **The forecaster's calibration is imperfect** on the amount of data used in `03_forecaster.ipynb` (measured 1-sigma coverage of ~41% against a ~68% target — a known, stated limitation, not hidden).
- **Control-loop latency** (`01_agent_overview.ipynb`, Section 10): every timestep would include a real LLM API call in a genuine deployment — a real system needs either a much faster model or a longer control interval than this simulation assumes.


## 5. Roadmap Extensions Not Yet Built

From `notebooks/00_overview.ipynb`'s roadmap, these remain open:

- Live dashboard with real-time traffic/allocation charts.
- The "agent reasoning" panel surfacing each decision's `reasoning` field live (the field already exists and is populated — `notebooks/01_agent_overview.ipynb`, Section 8 — it just isn't rendered anywhere yet).
- A manual fault/spike-injection control for live side-by-side demos.
- Running the full three-way comparison against `real_anthropic_llm_call()` instead of the reference heuristic, to get a true (not lower-bound) picture of the live agentic system's performance.


## 6. Summary

The project replaced an earlier opaque, purely-reactive PPO agent with a proactive, explainable, independently-verified pipeline: Digital Twin → Probabilistic Forecaster → LLM Planner → Safety Layer. On identical, spike-containing synthetic traffic, the resulting system reduced URLLC QoS violations relative to both a static baseline and a reactive legacy-behaviour agent, at a real and explicitly quantified eMBB throughput cost — with a safety layer that, under adversarial testing, never once let an unsafe plan through.

---
*This is the last notebook in the sequence. See `README.md` for the full repository structure and how to reproduce any result above.*
